In [ ]:
%pip install requests pandas folium -q

In [ ]:
import time
import requests
import pandas as pd

In [ ]:
# ---- EDIT THESE ----
ADDRESS = "City, state, country"   # your address, city, or landmark
RADIUS_KM = 5                          # search radius in kilometers

# Optional: you can skip geocoding entirely by setting coordinates directly but if you want to try it. Go ahead.
LATITUDE = None
LONGITUDE = None

# Which kinds of places to include — feel free to add/remove entries.
# "shop", "office" and "craft" tags are pulled in full (any value) separately below.
AMENITY_TYPES = [
    "restaurant",
    "cafe",
    "bakery",
    "bar",

    "clothing_store",
    "shoe_store",
    "jewelry_store",
    "electronics_store",
    "cell_phone_store",
    "furniture_store",
    "hardware_store",
    "home_goods_store",
    "grocery_store",
    "supermarket",
    "convenience_store",

    "beauty_salon",
    "hair_salon",
    "spa",
    "gym",

    "car_repair",
    "car_dealer",
    "motorcycle_dealer",

    
    "veterinary_care",

    "real_estate_agency",
    "insurance_agency",

    "travel_agency",
    "florist",
    "pet_store",
    "book_store",
    "gift_shop",

    "hotel",
    "lodging"
]

In [ ]:
 def geocode(address):
    url = "https://nominatim.openstreetmap.org/search"
    params = {"q": address, "format": "json", "limit": 1}
    
    headers = {"User-Agent": "local-business-finder-notebook (personal, non-commercial use)"}
    resp = requests.get(url, params=params, headers=headers, timeout=15)
    resp.raise_for_status()
    results = resp.json()
    if not results:
        raise ValueError(f"Could not find coordinates for '{address}'. Try a more specific address.")
    return float(results[0]["lat"]), float(results[0]["lon"])

if LATITUDE is None or LONGITUDE is None:
    LATITUDE, LONGITUDE = geocode(ADDRESS)

print(f"Search center: {LATITUDE}, {LONGITUDE}")

In [ ]:
# Several free public Overpass mirrors exist — if one is busy, we fall back to the next.
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
    "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
]
OVERPASS_HEADERS = {
    # Public instances require an identifiable client; update this if you publish the notebook.
    "User-Agent": "local-business-finder-notebook/1.0 (personal, non-commercial use)",
    "Accept": "application/json",
}

def build_query(lat, lon, radius_km, amenity_types):
    radius_m = int(radius_km * 1000)
    amenity_regex = "|".join(amenity_types)
    return f"""
    [out:json][timeout:60];
    (
      node["shop"](around:{radius_m},{lat},{lon});
      way["shop"](around:{radius_m},{lat},{lon});
      node["office"](around:{radius_m},{lat},{lon});
      way["office"](around:{radius_m},{lat},{lon});
      node["craft"](around:{radius_m},{lat},{lon});
      way["craft"](around:{radius_m},{lat},{lon});
      node["amenity"~"{amenity_regex}"](around:{radius_m},{lat},{lon});
      way["amenity"~"{amenity_regex}"](around:{radius_m},{lat},{lon});
    );
    out center tags;
    """

def run_overpass(query, attempts_per_endpoint=2):
    # Retry only transient conditions: overloaded servers, timeouts, and network errors.
    last_error = None
    for endpoint in OVERPASS_ENDPOINTS:
        for attempt in range(attempts_per_endpoint):
            try:
                resp = requests.post(
                    endpoint, data={"data": query}, headers=OVERPASS_HEADERS, timeout=90
                )
                resp.raise_for_status()
                return resp.json()
            except requests.RequestException as e:
                last_error = e
                status = getattr(e.response, "status_code", None)
                if status is not None and status not in {429, 500, 502, 503, 504}:
                    print(f"  {endpoint} rejected the request ({e}); trying the next mirror...")
                    break
                if attempt + 1 < attempts_per_endpoint:
                    delay = 3 * (attempt + 1)
                    print(f"  {endpoint} is temporarily unavailable ({e}); retrying in {delay}s...")
                    time.sleep(delay)
                else:
                    print(f"  {endpoint} failed ({e}); trying the next mirror...")
    raise RuntimeError(
        f"No Overpass server was available. Please wait a few minutes and run this cell again. Last error: {last_error}"
    )

query = build_query(LATITUDE, LONGITUDE, RADIUS_KM, AMENITY_TYPES)
print("Querying OpenStreetMap, this can take up to a minute...")
raw = run_overpass(query)
elements = raw.get("elements", [])
print(f"Found {len(elements)} raw features.")

In [ ]:
def get_coords(el):
    if el["type"] == "node":
        return el.get("lat"), el.get("lon")
    center = el.get("center", {})
    return center.get("lat"), center.get("lon")

def get_category(tags):
    for key in ["shop", "amenity", "office", "craft"]:
        if key in tags:
            return f"{key}={tags[key]}"
    return "unknown"

def get_address(tags):
    parts = [
        tags.get("addr:housenumber", ""),
        tags.get("addr:street", ""),
        tags.get("addr:city", ""),
    ]
    return " ".join(p for p in parts if p).strip()

WEB_PRESENCE_TAGS = [
    "website", "contact:website", "facebook", "contact:facebook", "contact:instagram",
]

rows = []
for el in elements:
    tags = el.get("tags", {})
    name = tags.get("name")
    if not name:
        continue  # skip unnamed features, not useful as leads
    lat, lon = get_coords(el)
    has_web = any(tags.get(k) for k in WEB_PRESENCE_TAGS)
    rows.append({
        "name": name,
        "category": get_category(tags),
        "address": get_address(tags),
        "phone": tags.get("phone") or tags.get("contact:phone", ""),
        "website": tags.get("website") or tags.get("contact:website", ""),
        "has_web_presence": bool(has_web),
        "lat": lat,
        "lon": lon,
    })

df = pd.DataFrame(rows).drop_duplicates(subset=["name", "lat", "lon"]).reset_index(drop=True)
print(f"{len(df)} named businesses found.")
df.head()

In [ ]:
df_no_website = (
    df[~df["has_web_presence"]]
    .drop(columns=["has_web_presence", "website"])
    .reset_index(drop=True)
)
print(f"{len(df_no_website)} of {len(df)} businesses have no website or social page listed on OpenStreetMap.")
df_no_website

In [ ]:
output_file = "businesses_no_website.csv"
df_no_website.to_csv(output_file, index=False)
print(f"Saved {len(df_no_website)} businesses to {output_file}")

In [ ]:
import folium

m = folium.Map(location=[LATITUDE, LONGITUDE], zoom_start=14)
folium.Marker(
    [LATITUDE, LONGITUDE], tooltip="Search center", icon=folium.Icon(color="blue")
).add_to(m)

for _, row in df_no_website.iterrows():
    if pd.notna(row["lat"]) and pd.notna(row["lon"]):
        folium.Marker(
            [row["lat"], row["lon"]],
            tooltip=row["name"],
            popup=f"{row['name']} ({row['category']})",
            icon=folium.Icon(color="red", icon="info-sign"),
        ).add_to(m)

m

In [ ]:
## Notes & good practice
- Uses only free, keyless services: **Nominatim** (geocoding) and **Overpass API** (OSM data). No sign-up, no billing, no credit card.
- OSM data is community-maintained — "no website tag" isn't the same as "definitely no website." Spot-check a handful of results before treating this as final.
- Be considerate of the free Overpass servers: a few queries a minute is fine, running huge radii in a tight loop isn't — that's why the notebook tries multiple mirrors instead of hammering one.
- If you're planning outreach (calls, texts, emails) from this list, check India's rules on unsolicited commercial communication (TRAI/DND registry) before reaching out at scale.